# Constructing a Goal-Based Agent using an LLM
**Course:** Artificial Intelligence (CS F407)
**Topic:** Goal-Based Agent for Warehouse Navigation

This notebook implements the warehouse navigation environment, agent architecture, BFS path planning algorithm, and ASCII map rendering.

In [1]:
from collections import deque
from typing import List, Tuple, Optional, Dict, Set

WAREHOUSE_MAP = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################"
]
print("
".join(WAREHOUSE_MAP))

In [2]:
class WarehouseEnvironment:
    def __init__(self, grid_map: List[str]):
        self.grid = [row.strip() for row in grid_map if row.strip()]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0]) if self.rows > 0 else 0
        self.start_pos = None
        self.goal_pos = None
        self.obstacles = set()
        self._parse()

    def _parse(self):
        for r in range(self.rows):
            for c in range(self.cols):
                char = self.grid[r][c]
                if char == "S": self.start_pos = (r, c)
                elif char == "G": self.goal_pos = (r, c)
                elif char == "#": self.obstacles.add((r, c))

    def is_valid(self, pos: Tuple[int, int]) -> bool:
        r, c = pos
        return 0 <= r < self.rows and 0 <= c < self.cols and pos not in self.obstacles

    def get_actions(self, pos: Tuple[int, int]):
        r, c = pos
        moves = [("Up", (r - 1, c)), ("Down", (r + 1, c)), ("Left", (r, c - 1)), ("Right", (r, c + 1))]
        return [(act, p) for act, p in moves if self.is_valid(p)]

env = WarehouseEnvironment(WAREHOUSE_MAP)
print(f"Start: {env.start_pos}, Goal: {env.goal_pos}, Obstacles: {len(env.obstacles)}")

In [3]:
class GoalBasedAgent:
    def __init__(self, env: WarehouseEnvironment):
        self.env = env
        self.start = env.start_pos
        self.goal = env.goal_pos

    def search(self):
        queue = deque([(self.start, [self.start])])
        visited = {self.start}
        expanded = 0
        while queue:
            curr, path = queue.popleft()
            expanded += 1
            if curr == self.goal:
                return path, expanded
            for _, nxt in self.env.get_actions(curr):
                if nxt not in visited:
                    visited.add(nxt)
                    queue.append((nxt, path + [nxt]))
        return None, expanded

    def render(self, path):
        path_set = set(path) if path else set()
        lines = []
        for r in range(self.env.rows):
            row = []
            for c in range(self.env.cols):
                p = (r, c)
                if p == self.start: row.append("S")
                elif p == self.goal: row.append("G")
                elif p in path_set: row.append("*")
                elif p in self.env.obstacles: row.append("#")
                else: row.append(".")
            lines.append("".join(row))
        return "
".join(lines)

agent = GoalBasedAgent(env)
path, expanded = agent.search()
print(f"Path Found: {path is not None}")
print(f"Steps: {len(path) - 1}, States Expanded: {expanded}")
print("
Rendered Path:")
print(agent.render(path))